In [5]:
import debugai
! python3 -m spacy download en_core_web_sm
print(debugai.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 33.0 MB/s  0:00:0027.2 MB/s eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
0.2.0


In [6]:
result = debugai.analyze(
    prompt="What is the refund policy for electronics?",
    output="Electronics can be returned within 90 days for a full cash refund, no receipt needed.",
    chunks=["Store hours are 9am to 5pm.", "Parking is behind the building."],
    similarity_scores=[0.42, 0.40],
    temperature=0.2,
    explain_with_llm=False,
)

print("Failure:", result["primary"]["failure"])       # retrieval_failure
print("Confidence:", result["primary"]["confidence"]) # 0.95
print("Fix:", result["primary"]["fix"])
print("Healthy:", result["healthy"])                  # False

Failure: retrieval_failure
Confidence: 0.95
Fix: Re-chunk source documents with an entity-aware strategy, tune the retriever / embedding model, or expand the knowledge base.
Healthy: False


In [7]:
from openai import OpenAI

client = debugai.wrap_llm(
    OpenAI(),   # reads OPENAI_API_KEY from env
    on_diagnosis=lambda d: print("Diagnosis:", d["primary"]["failure"] if not d["healthy"] else "healthy"),
)

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello"}],
)
print(resp.choices[0].message.content)
client.debugai.flush()

Hello! How can I assist you today?Diagnosis: healthy



In [4]:
resp = debugai.completion("gpt-4o-mini", messages=[
    {"role": "user", "content": "What is 2+2?"}
])
print(resp.text, resp.cost_usd, resp.latency_ms)
print(debugai.metrics.snapshot())




2 + 2 equals 4. 7e-06 1669
{'requests': 1, 'prompt_tokens': 8, 'completion_tokens': 9, 'total_tokens': 17, 'cost_usd': 7e-06, 'failures': 0, 'cache_hits': 0, 'cache_misses': 1, 'latency_p50_ms': 2246.0, 'latency_p95_ms': 2246.0, 'by_model': {'gpt-4o-mini': {'requests': 1, 'prompt_tokens': 8, 'completion_tokens': 9, 'total_tokens': 17, 'cost_usd': 7e-06, 'failures': 0, 'cache_hits': 0, 'cache_misses': 1, 'latency_p50_ms': 2246.0, 'latency_p95_ms': 2246.0}}}


In [8]:
r = debugai.analyze(
    prompt="What does Section 4 of the contract say?",
    output="Section 4 requires arbitration in Delaware under the Marbury Clause and a $50,000 penalty.",
    chunks=["Section 4 covers confidentiality.", "Governed by California law."],
    similarity_scores=[0.66, 0.59],
    temperature=0.75,
    explain_with_llm=False,
)

print(r["primary"]["failure"])     # hallucination
print(r["primary"]["confidence"])  # 0.95
print(r["primary"]["fix"])

hallucination
0.95
Add grounding constraints to the system prompt (answer only from provided context; cite sources; say 'not found' when unsupported).
